# EDA v2: E-CUP Matching

Новый проход по данным после `01_eda` / `02_eda_deep`. Полные цифры и выводы: [`insights.md`](../insights.md). Тяжёлый пересчёт: `eda_v2_compute.py` → `eda_v2_stats.json`.

Этот ноутбук — живой просмотр пар и атрибутов на human-срезе.

In [ ]:
import json, collections
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_rows", 40)
DATA = Path("..")
STATS = Path("eda_v2_stats.json")

In [ ]:
items = pd.read_parquet(DATA / "items_human.parquet")
matches = pd.read_parquet(DATA / "matches.parquet")
m = (
    matches.merge(items.add_suffix("_1"), left_on="id1", right_on="id_1")
           .merge(items.add_suffix("_2"), left_on="id2", right_on="id_2")
)
print(items.shape, matches.shape, m.shape)
print("pos-rate", m.target.mean().round(4))
print("category mismatch", (m.category_1 != m.category_2).sum())

## Что смотреть

1. Human и LLM не пересекаются даже по id товаров — LLM это другой корпус, не псевдо-лейблы тех же пар.
2. Пары уже lookalike: JW 0.71 vs 0.68. Лексика слабая.
3. Одинаковое имя в моде ≈ негатив (pos-rate 9%), вне моды ≈ позитив (67%).
4. Пересечение артикулов → 90% pos, но это 0.4% пар. Конфликт цвета → сильный neg.
5. Ювелирка / одежда / обувь — дыры макро-метрики (LGBM AP ≈ 0.27–0.28).
6. 34k уникальных JSON-ключей; `бренд` в одежде пустой — значение в `бренд в одежде и обуви`.

## Примеры пар по категории и классу

In [ ]:
def show_pairs(cat=None, target=None, n=8, query=None):
    df = m
    if cat is not None:
        df = df[df.category_1 == cat]
    if target is not None:
        df = df[df.target == target]
    if query:
        q = query.lower()
        df = df[df.name_1.str.contains(q, case=False, na=False) | df.name_2.str.contains(q, case=False, na=False)]
    cols = ["target", "category_1", "name_1", "name_2"]
    return df[cols].sample(min(n, len(df)), random_state=0) if len(df) else df[cols]

show_pairs(cat="Ювелирные изделия", target=1, n=6)

In [ ]:
show_pairs(cat="Одежда", target=0, n=6)

In [ ]:
# Точное совпадение названия при target=0 — почти всегда мода с пустым SKU в имени
eq_neg = m[(m.name_1 == m.name_2) & (m.target == 0)]
print(len(eq_neg), eq_neg.category_1.value_counts().head(8).to_string())
eq_neg[["category_1", "name_1"]].sample(8, random_state=1)

## Атрибуты одного товара

In [ ]:
def peek(cat, k=1):
    row = items[items.category == cat].sample(k, random_state=2).iloc[0]
    obj = json.loads(row.attributes)
    print(row.category, "|", row["name"])
    print(f"{len(obj)} keys:", list(obj)[:20])
    return pd.Series(obj)

peek("Автотовары")

In [ ]:
peek("Одежда")

## Готовые агрегаты из `eda_v2_stats.json`

In [ ]:
s = json.loads(STATS.read_text())
print("runtime", s["runtime_sec"], "s")
print("LLM item overlap with human", s["llm"]["human_items_also_in_llm_pairs"])
print("article intersection pos-rate", s["struct_lifts"]["art_inter"]["pos_rate_when"])
pd.DataFrame(s["by_category"])[["category", "n_pairs", "pos_rate", "mean_tok_pos", "mean_tok_neg"]].sort_values("pos_rate")